# 関数定義

In [ ]:
import requests
import json
import re
from datetime import datetime

def get_game_id(game_name):
    """ゲーム名からゲームIDを取得する"""
    base_url = "https://www.speedrun.com/api/v1/games"
    params = {"name": game_name}
    response = requests.get(base_url, params=params)
    data = response.json()
    
    if 'data' in data and data['data']:
        return data['data'][0]['id']
    return None

def get_category_ids(game_id):
    """ゲームIDからカテゴリIDを取得する"""
    base_url = f"https://www.speedrun.com/api/v1/games/{game_id}/categories"
    response = requests.get(base_url)
    data = response.json()

    if 'data' in data:
        return {category['id']: category['name'] for category in data['data']}
    return {}

def get_all_runs(game_id, category_id):
    """ゲームIDとカテゴリIDから全てのランの詳細を取得する"""
    base_url = f"https://www.speedrun.com/api/v1/leaderboards/{game_id}/category/{category_id}"
    response = requests.get(base_url)
    data = response.json()

    if 'data' in data and 'runs' in data['data']:
        return data['data']['runs']
    return []

def save_to_json(data, filename):
    """データをJSONファイルに保存する"""
    with open(filename, 'w') as json_file:
        json.dump(data, json_file, indent=4)

def load_data(filepath):
    """指定されたパスからJSONファイルをロードする"""
    try:
        with open(filepath, 'r') as file:
            data = json.load(file)
        return data
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None
    except json.JSONDecodeError:
        print(f"Error decoding JSON from file: {filepath}")
        return None

def parse_duration(duration_str):
    """ISO 8601 Duration (例: "PT30M45.5S") を秒に変換"""
    time_units = {'H': 3600, 'M': 60, 'S': 1}
    total_seconds = 0
    matches = re.findall(r'(\d+\.?\d*)([HMS])', duration_str)
    for value, unit in matches:
        total_seconds += float(value) * time_units[unit]
    return total_seconds

# メタデータ取得

In [ ]:
game_name = input("ゲーム名を入力してください: ")
game_id = get_game_id(game_name)

if not game_id:
    print(f"{game_name} のゲームIDが見つかりませんでした。")
else:
    category_data = get_category_ids(game_id)
    all_runs = []
    
    for category_id in category_data.keys():
        runs = get_all_runs(game_id, category_id)
        all_runs.extend(runs)

    filename = f"{game_name.replace(' ', '_').lower()}_speedrun_info.json"
    save_to_json(all_runs, filename)
    print(f"総ラン数: {len(all_runs)}件が'{filename}'に保存されました。")

# 各カテゴリの歴代WR取得

In [ ]:
def process_data(data):
    """カテゴリごとにランのデータを整理"""
    categories = {}

    for run_data in data:
        run = run_data.get('run', {})
        category = run.get('category')
        date_str = run.get('date')
        run_time = run.get('times', {}).get('primary')
        videos = run.get('videos', {})
        video_links = [link.get('uri') for link in videos.get('links', [])] if videos else []

        run_id = run.get('id')

        if date_str is None or run_time is None:
            print(f"Missing date or time in run data: {run_id}")
            continue

        try:
            run_date = datetime.strptime(date_str, '%Y-%m-%d')
        except ValueError as e:
            print(f"Date format error for run ID {run_id}: {date_str} ({e})")
            continue

        run_time_seconds = parse_duration(run_time)

        if category not in categories:
            categories[category] = []

        categories[category].append({
            'date': run_date,
            'time': run_time_seconds,
            'video_links': video_links,
            'run_id': run_id
        })

    for category in categories:
        categories[category].sort(key=lambda x: x['date'])

    results = {}
    for category, runs in categories.items():
        results[category] = []
        for run in runs:
            if not results[category] or run['time'] < results[category][-1]['time']:
                results[category].append(run)

    return results

data = load_data(filename)
if data is not None:
    categories = process_data(data)
    category_names = category_data  # get_category_ids() から取得済み

    def write_video_links(categories, category_names):
        """各カテゴリの動画リンクをテキストファイルに書き出す"""
        for category_id, records in categories.items():
            category_name = category_names.get(category_id, 'Unknown_Category')
            filename = f"{category_name.replace(' ', '_')}_video_links.txt"
            with open(filename, 'w') as file:
                for record in sorted(records, key=lambda x: x['date']):
                    if record['video_links']:
                        file.write(f"{record['date'].strftime('%Y-%m-%d')}:\n")
                        for link in record['video_links']:
                            file.write(f"  {link}\n")
                    else:
                        print(f"Run ID {record['run_id']} has no video link.")

    write_video_links(categories, category_names)
    print("動画リンクの書き出しが完了しました。")